In [0]:
from main import UC_TOOL_NAMES,VECTOR_SEARCH_TOOL

import mlflow
from mlflow.models.resources import DatabricksFunction
from pkg_resources import get_distribution


resources = []

resources.extend(VECTOR_SEARCH_TOOL.resources)


for tool_name in UC_TOOL_NAMES:
    resources.append(DatabricksFunction(function_name=tool_name))


with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="foodly-ai-assistant",
        code_paths=["helpers.py"],
        python_model='main.py',
        pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"mlflow=={get_distribution('mlflow').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            f"langgraph=={get_distribution('langgraph').version}",
             f"langgraph-supervisor=={get_distribution('langgraph-supervisor').version}",
        ],
        resources=resources,
    )

🔗 View Logged Model at: https://8259551295636485.5.gcp.databricks.com/ml/experiments/16a101a6d89648ecbd38b4d419a761c3/models/m-f02b87f0411649cd91ee96ff20ef81cc?o=8259551295636485
2026/06/22 10:48:42 INFO mlflow.pyfunc: Predicting on input example to validate output

### Pre-deployment agent validation


In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/foodly-ai-assistant",
    input_data={"input": [{"role": "user", "content": "Tell me about foodly return policy"}]},
    env_manager="uv",
)

In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model
catalog = "agents"
schema = "main"
model_name = "foodly-ai-assistant"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)

Successfully registered model 'agents.main.foodly-ai-assistant'.

In [0]:
from databricks import agents

agents.deploy(
    UC_MODEL_NAME,
    uc_registered_model_info.version,
    scale_to_zero=True,
    tags={"name": "foodly-ai-assistant-endpoint","creator" : "anirvan-sen"},
)


    Deployment of agents.main.foodly-ai-assistant version 1 initiated.  This can take up to 15 minutes and the Review App & Query Endpoint will not work until this deployment finishes.

    View status: https://8259551295636485.5.gcp.databricks.com/ml/endpoints/agents_agents-main-foodly-ai-assistant
    Review App: https://8259551295636485.5.gcp.databricks.com/ml/review-v2/5bc6093206f047f5b2faf873d99b25c0/chat
    Monitor: https://8259551295636485.5.gcp.databricks.com/ml/experiments/16a101a6d89648ecbd38b4d419a761c3?compareRunsMode=TRACES
Deployment(model_name='agents.main.foodly-ai-assistant', model_version='1', endpoint_name='agents_agents-main-foodly-ai-assistant', served_entity_name='agents-main-foodly-ai-assistant_1', query_endpoint='https://8259551295636485.5.gcp.databricks.com/serving-endpoints/agents_agents-main-foodly-ai-assistant/served-models/agents-main-foodly-ai-assistant_1/invocations', endpoint_url='https://8259551295636485.5.gcp.databricks.com/ml/endpoints/agents_agents-main-foodly-ai-assistant', review_app_url='https://8259551295636485.5.gcp.databricks.com/ml/review-v2/5bc6093206f047f5b2faf873d99b25c0/chat')

[](url)

[](url)

[](/Users/jagritipriyadarshani@gmail.com/build-ai-agent-that-works/build-ai-agent-to-work/main/4.end to end project/foodly_ai_support/foodly_ai_assistant.png)

[](url)